# Eurobarometer Data Extraction

As mentioned before, the valuable Eurobarometer data comes in a messy `.xlsx` format, which is not usable for the purposes of data analysis. The idea is to come up with a solution for a messy Excel file with multiple tabs, because the newest Eurobarometer Surveys are only presented in this format. The data requires a lot of transformation.

**The problems**:
- impossible to export a given Excel file to `.csv` because of multiple tabs; the solution is to read the file in a loop, read each tab and write it into separated `.csv` (to merge them in the future);
- each tab should be edited: it has a lot of unnecessary info (like text, questions in French, etc); these have to be removed;
- structure of the tabs is not the same (e.g., there are tabs with 'open questions' or similar);
- column Total (easy to remove in comparison to other transformation);
- rows are duplicated: the info fiven in absolute numbers (number of informants who gave a certain answer) and percentage (of people in a given country who answered the question in a particular way); the rows with absolute numbers should be removed;
- countries are columns but should be rows;
- each indicator (e.g., trust to a national government) have multiple dimensions (fully trust (1), somewhat trust (2), neutral (3), somewhat do not trust (4), absolutely don't trust (5), don't know (6) etc); the amount of answers is not the same for each question; it should be adjusted:
    - option 1: keep everything (if it's easier); than each indicator will be converted to many (Depending on how many alternatives the survey participants were given);
    - option 2: drop those who don't know or neutral and keep only those who are 'positive' (e.g., take those who are fully trust + somewhat trust and make a new indicator);
    - option 3: probably there are other options; but the idea is to get ONE number per indicator per country
- YEAR column: has to be added.

There are several tasks which need to be adressed in this notebook:
- downloading raw data for three Eurobarometers (metadata file with links/names of the tables will be provided);
- transforming `.xlsx` files into machine-readable `.csv` files;
- uploading them to PostgreSQL.

In the current notebook all the examples will be implemented on a `.xlsx` file, representing Special Eurobarometer "Digital Decade" (no. SP566, 2025).

## 1. Reading Tabs | Example

I'm going to work with the EU countries only, but this part could be skipped by those who want to include in the analysis all the data:

In [ ]:
eu_countries = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 
    'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 
    'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

We can get several tabs from the file are check how they look like / try to read them and transform into a dataframe, which will allow us saving the tab later as a `.csv` file.

I will read the following tabs:
- 'B': List of countries,
- 'QE1_5': Engaging in democratic life via digital technology, importance
- 'QE2': Digitalization of daily life: makes it easier or more difficult
- 'C2': Political Interest Index

In [ ]:
import pandas as pd
import re

# tabs ro read:
specific_tabs = ['B', 'QE1_5', 'QE2', 'C2']

# dictionary of dataframes: 
dict_of_dfs = pd.read_excel('./eurobarometer_data/eb_sp566.xlsx', sheet_name=specific_tabs, header=8)

# reading the tabs separately and printing their names:
for sheet_name, df in dict_of_dfs.items():
    print(f"Reading tab: {sheet_name}")

#### B: Countries
Now we can take a closer look at the `B` (Countries) tab:

In [ ]:
df_countries = dict_of_dfs['B']
df_countries.head(5)

This dataframe already looks pretty messy:
- columns that we don't need (e.g., "Back to content")
- multiple rows that do not serve any purpouse (all the info is avaliable in row 1)
- etc.

We can get rid of all of it and save a clean dataframe as a `.csv` file (in case we need to know, how many participants from each country answered the Eurobarometer questions).

In [ ]:
# start over:
df_countries = dict_of_dfs['B']

# drop useless columns:
df_countries = df_countries.drop(columns=['<<Back to content', 'UE27\nEU27'])

# grab only one row and make it to pivot the table
df_countries = df_countries.iloc[[0]].T
df_countries = df_countries.iloc[1:]
df_countries = df_countries.reset_index()

# change the column names:
df_countries = df_countries.rename(columns={
    'index': 'country',
    0: 'num_ind'
})

# drop the non-EU countries:
df_countries = df_countries[df_countries['country'].isin(eu_countries)]

# check the result:
df_countries.head(10)
# df_countries.shape

# write to a file:
# df_countries.to_csv('B.csv', index=False)

#### C2: Political Interest Index

In [ ]:
# get the data, drop obsolete columns:
df_polit_ind = dict_of_dfs['C2'].drop(columns=['<<Back to content', 'UE27\nEU27'])
df_polit_ind.rename(columns={'Unnamed: 1': 'intensity_c2'}, inplace=True)

# map the symbols in rows to replace them
symbol_mapping = {'Total': 'total', '+ +': 'high', '+': 'medium', '-': 'low', '- -': 'very_low'}
df_polit_ind['intensity_c2'] = df_polit_ind['intensity_c2'].replace(symbol_mapping)

# find numeric columns and make them numeric for real:
num_cols = df_polit_ind.columns.drop('intensity_c2')
df_polit_ind[num_cols] = df_polit_ind[num_cols].apply(pd.to_numeric, errors='coerce')

# remove rows with absolute numbers (but keep %%)
df_polit_ind = df_polit_ind[df_polit_ind['BE'] <= 1]

# set the index:
df_polit_ind.set_index('intensity_c2', inplace=True)

# flip the table:
df_polit_ind = df_polit_ind.T

# filter to keep the EU countries only:
df_polit_ind = df_polit_ind[df_polit_ind.index.isin(eu_countries)]

# clear the indexing:
df_polit_ind.index.name = 'country'
df_polit_ind = df_polit_ind.reset_index()
df_polit_ind.columns.name = None

df_polit_ind

# save as .csv if needed:
# df_polit_ind.to_csv('polit_interest_index.csv', index=False)

#### 'QE2' | Digitalization and better life

In [ ]:
df_easy_life = dict_of_dfs['QE2'].drop(columns={'<<Back to content','UE27\nEU27'})
num_cols = df_easy_life.columns.drop('Unnamed: 1')
df_easy_life[num_cols] = df_easy_life[num_cols].apply(pd.to_numeric, errors='coerce')
df_easy_life = df_easy_life[df_easy_life['BE'] <= 1]
df_easy_life.set_index('Unnamed: 1',inplace=True)
df_easy_life = df_easy_life.T
df_easy_life.index.name = 'country'
df_easy_life = df_easy_life[df_easy_life.index.isin(eu_countries)]
df_easy_life.reset_index(inplace=True)
df_easy_life.columns.name = None
df_easy_life.columns = [str(col).lower().replace(' ', '_').replace("'","") for col in df_easy_life.columns]
df_easy_life

# df_easy_life.to_csv('easier_life.csv', index=False)

#### QE1_5 | Engaging in Democratic Life via Digital Technologies

In [ ]:
df_engage = dict_of_dfs['QE1_5'].drop(columns={'<<Back to content','UE27\nEU27'})
num_cols = df_engage.columns.drop('Unnamed: 1')
df_engage[num_cols] = df_engage[num_cols].apply(pd.to_numeric, errors='coerce')
df_engage = df_engage[df_engage['BE'] <= 1]
df_engage.set_index('Unnamed: 1',inplace=True)
df_engage = df_engage.T
df_engage.index.name = 'country'
df_engage = df_engage[df_engage.index.isin(eu_countries)]
df_engage.reset_index(inplace=True)
df_engage.columns.name = None
df_engage.columns = [str(col).lower().replace(' ', '_').replace("'","") for col in df_engage.columns]
df_engage

# df_engage.to_csv('df_engage.csv', index=False)

Now, I can read all the tabs from an Excel file and save them as a dictionary of dataframes + make some pre-cleaning (mainly dropping two tabs, content and countries). This code is a prototype for a future function.

In [ ]:
# read all the sheets as dictionary of dataframes:
dict_of_dfs = pd.read_excel('./eurobarometer_data/eb_sp566.xlsx', sheet_name=None, header=8)

# get rid of the front page and country list:
front_page = next(iter(dict_of_dfs))
del dict_of_dfs[front_page]
country_page = next(iter(dict_of_dfs))
del dict_of_dfs[country_page]

# get all the keys (names of the sheets):
dict_of_dfs.keys()

Here is a clean version of the function, responsible for cleaning the data only.

In [ ]:
def clean_eurobarometer(dict_of_dfs, sheet:str):
    """This function helps to deal with messy data from Eurobarometer.
    It takes two arguments: 
    - a dictionary of dataframes (dict_of_dfs)
    - a name of a tab from Excel file (string).
    Cleans it, pivots it, re-assigns data types and removes obsolete data.
    It also drops all the countries which are not EU.
    It returns a clean data frame."""

    eu_countries = [
        'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 
        'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 
        'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
    ]

    try: 
        df = dict_of_dfs[sheet].drop(columns={'<<Back to content','UE27\nEU27'})
        num_cols = df.columns.drop('Unnamed: 1')
        df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
        df = df[df['BE'] <= 1]
        df.set_index('Unnamed: 1',inplace=True)
        df = df.T
        df.index.name = 'country'
        df = df[df.index.isin(eu_countries)]
        df.reset_index(inplace=True)
        df.columns.name = None
        df.columns = [
            'country' if col == 'country' 
            else f'{sheet}_'.lower() + str(col).lower().strip().replace(' ', '_').replace("'", "").replace(":", "_").replace(",","") 
            for col in df.columns
        ]
        return df
    except Exception as e:
        print(f"Something went wrong. You need to deal with: {e}.")

# df_easy_life.to_csv('easier_life.csv', index=False)

In [ ]:
# test:
clean_eurobarometer(dict_of_dfs, 'QE1_5')

Merging two dfs, which are being cleaned:

In [ ]:
# test merging two random tabs:
df_qe1_5=clean_eurobarometer(dict_of_dfs, 'QE1_5')
df_qe9_6=clean_eurobarometer(dict_of_dfs, 'QE9_6')

In [ ]:
df_joined_test = pd.merge(df_qe1_5, df_qe9_6, on='country', how='inner')

In [ ]:
df_joined_test

So the merge works, now I need to come up with a solution of merging ALL the tabs in one table. Probably. To make one `.csv` file from these all tabs in the end.

**Nota Bene**  
It's absolutely not recommended to merge different multiple dataframes in a loop, as I originally thought. There is a specific function for it called `reduce` which comes from `functools` library. I will use this one.

Get all the dfs in a list of dfs:

In [ ]:
all_tabs = []
for sheet_name, df in dict_of_dfs.items():
    # print(f"Processing tab {sheet_name}...")
    df_clean = clean_eurobarometer(dict_of_dfs, sheet_name)
    #print(type(df_clean))
    # df_clean.to_csv(f"{sheet_name}.csv", index=False) # save them as separated csvs if needed
    all_tabs.append(df_clean)

Merge them together using `reduce`:

In [ ]:
from functools import reduce 

In [ ]:
# all dfs are taken from a list and then merged with lambda function
df_final = reduce(lambda left, right: pd.merge(left, right, on=['country'], how='inner'), all_tabs)

# check the number of rows and columns:
df_final.shape

In [ ]:
# save it to csv to see the result:
df_final.to_csv('final_test_2.csv',index=False)

I will need to create table dynamicaaly on PostgreSQL. For this I will use the same approach as with the Eurostat data but I will turn it to a function too.
Test the same approach with table structures (as with Eurostat data):

In [ ]:
raw_columns =  df_qe1_5.columns.tolist()
structure_elements = []
table_structure = ''
for col in raw_columns:
    if str(col) == 'country':
        structure_elements.append(f'"{col}" varchar')
    else:
        structure_elements.append(f'"{col}" numeric')
table_structure = ', '.join(structure_elements)

Now, the idea is that the final `.csv` is actually not going to be saved but is going to go directly to PostgreSQL database. For this, I need to repeat the same steps as for data retrieval from Eurostat website.

After 'assembling' everything in one `.py` script, I tested it one one tab from the same Excel file. It worked out. The test went to the `__main__` and looked this way:

``` python
if __name__ == "__main__":
# ORIGINAL TEST:
# this worked!
    path_test = './eurobarometer_data/eb_sp566.xlsx'
    dict_test = read_eurobarometer(path_test)
    map_path = 'mapping_columns.csv'
    df_test = clean_eurobarometer(dict_test, 'QE1_5',map_path)
    
    load_dotenv(dotenv_path='.env')
    test_name = os.getenv('DB_NAME')
    test_user = os.getenv('DB_USER')
    connection_string_test = f'dbname = {test_name} user = {test_user}'
    table_test = 'eurobarometer_test'

    upload_to_postgres(df_test, connection_string_test, table_test)
```

While trying to run the code on all tabs, I encountered a problem with the length and content of column names in the Excel files. These are two main problems:
- length (PostgreSQL restricts the length of column names to 63 characters); the nature of columns in Eurobarometer (the way I designed it) is that they are converted from rows; and rows have the entire answer to a question, for example: "MARRIED OR REMARRIED: Living with the children of this marriage and of a previous marriage"
- partial repetiton of terms (since PostgreSQL import will cut column names that are too long, they will match to each other); example:
    - 'd7_married_or_remarried__living_with_the_children_of_this_marri' for two columns (children of this marriage and children of this marriage and previous marriage)

To deal with it, I had to create a `mapping_columns.csv` file, which helps to alter column names.

Then I decided to make one more test with more complex tab which would require a lot more column names editing and mapping:

```python
if __name__ == "__main__":

    ADDITIONAL TEST on D7 with Column Names Mapping:
    path_test = './eurobarometer_data/eb_sp566.xlsx'
    map_path = 'mapping_columns.csv'
    dict_test = read_eurobarometer(path_test)
    df_test = clean_eurobarometer(dict_test, 'D7', map_path)

    load_dotenv(dotenv_path='.env')
    test_name = os.getenv('DB_NAME')
    test_user = os.getenv('DB_USER')
    connection_string_test = f'dbname = {test_name} user = {test_user}'
    table_test = 'eurobarometer_d7_test'

    upload_to_postgres(df_test, connection_string_test, table_test)
```


Now my task is to re-write the `.py` script in a manner it will loop over *all* tabs in the Excel file, merge them using `reduce` and put them to PostgreSQL. Final run:
```python
    
if __name__ == "__main__":
    # FINAL RUN:
    path = './eurobarometer_data/eb_sp566.xlsx'
    dict_dfs = read_eurobarometer(path)

    all_tabs = []
    map_path = 'mapping_columns.csv'
    for sheet_name, df in dict_dfs.items():
        df_clean = clean_eurobarometer(dict_dfs,sheet_name,map_path)
        all_tabs.append(df_clean)

    df_final = reduce(lambda left, right: pd.merge(left, right, on=['country'], how='inner'), all_tabs)
    df_final['year']=2025

    load_dotenv(dotenv_path='.env')
    db_name = os.getenv('DB_NAME')
    db_user = os.getenv('DB_USER')
    connection_string_final = f'dbname = {db_name} user = {db_user}'
    table_name = 'eurobarometer_sp566'
    upload_to_postgres(df_final, connection_string_final, table_name)
```